# Sub-trajectory balance pilot on Kaggle

This runs the PI's next step from the 25 Nov meeting. Trajectory balance is replaced by the modified sub-trajectory balance of Hu et al., "Amortizing intractable inference in large language models" (ICLR 2024), with the log reward log p(Z | X) + log p(right answer | X Z) from the untuned model.

Two runs share the session, one on each T4, for 80 steps on Qwen3-1.7B:
- `prelim-SubTB-wait-s0` learns from the model's own answers and from answers forced to continue after " Wait".
- `prelim-SubTB-own-s0` learns from the model's own answers only.

The test matches the 25 Nov runs, so the results line up with T4 (GRPO with "Wait" answers) in `results/2026-11-25-kaggle-prelim.md`. `docs/subtb-pilot.md` lists every setting next to the paper's.

**Settings:** Accelerator **GPU T4 x2**, Internet **on**.

**Run it with Save Version, then Save & Run All (Commit)**, so it keeps running after you close the browser. If a run would hit Kaggle's 12-hour limit, it stops early, evaluates and saves. When it finishes, download `bundles/prelim-subtb-*.tar.gz` from the version's Output tab.

Read the "Reward check" table in the summary first: it shows whether the untuned model's reward ranks right answers above wrong ones.

In [ ]:
PAIR = "subtb"  # sub-trajectory balance with and without "Wait" answers (docs/subtb-pilot.md)
REPO_URL = "https://github.com/amitkov-sudo/grpo-vs-gfn-self-correction.git"

!test -d repo || git clone -q $REPO_URL repo
%cd repo
!git pull -q && git log -1 --format="commit %h, %cd"

In [ ]:
# Installs the requirements, runs the offline checks, then trains both runs side by side (80 steps, up to 11.5 hours).
!PAIR=$PAIR bash scripts/kaggle_prelim.sh

In [ ]:
import glob, json, os
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

display(Markdown(open("outputs/prelim-summary.md").read()))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
for path in sorted(glob.glob("outputs/prelim-*/log.jsonl")):
    evals = [r["eval"] for r in map(json.loads, open(path)) if "eval" in r]
    run = os.path.basename(os.path.dirname(path))
    steps = [e["step"] for e in evals]
    ax1.plot(steps, [e["fix_rate"] for e in evals], marker="o", label=run)
    ax2.plot(steps, [e["false_alarms"] for e in evals], marker="o", label=run)
ax1.set(xlabel="training step", ylabel="fix rate on 60 wrong starts (greedy)")
ax2.set(xlabel="training step", ylabel="false alarms on 60 correct starts")
ax1.legend()
plt.tight_layout()
plt.show()